In [ ]:
import os
import warnings
import logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings("ignore", category=UserWarning, module="keras.losses")
logging.getLogger("transformers").setLevel(logging.ERROR)
from transformers import AutoModelForImageClassification, ViTImageProcessor, pipeline
from PIL import Image

# Доп. словарь, для улучшения читаемости:
LABEL_TRANSLATION = {
    "AK": "AK (Актинический кератоз)",
    "BCC": "BCC (Базальноклеточная карцинома)",
    "BKL": "BKL (Доброкачественный кератоз)",
    "DF": "DF (Дерматофиброма)",
    "MEL": "MEL (Меланома)",
    "NV": "NV (Меланоцитарный невус)",
    "SCC": "SCC (Плоскоклеточная карцинома)",
    "VASC": "VASC (Сосудистое поражение)",
    "Unknown": "Unknown (Неизвестный класс)"
}

# Словарь для числовых меток:
LABEL_MAP = {
    "AK": 0, "BCC": 1, "BKL": 2, "DF": 3,
    "MEL": 4, "NV": 5, "SCC": 6, "VASC": 7
}

def load_model(model_path): # Загрузка модели
    model = AutoModelForImageClassification.from_pretrained(model_path)
    print(f"Модель загружена из {model_path}")
    return model

def evaluate_single_image(image_path, model_path): # обработка одного фото
    model = load_model(model_path) # Загрузка модели и процессора
    processor = ViTImageProcessor.from_pretrained(model_path)
    try:
        image = Image.open(image_path).convert("RGB") # Открытие и преобразование изображения + классификация
        classifier = pipeline("image-classification",
                            model=model,
                            feature_extractor=processor)

        result = classifier(image)[0]
        label_key = result['label'].split('_')[-1]  # Обработка метки, например: 'AK' из 'LABEL_0'
        # Получаем, к примеру: "AK (Актинический кератоз)":
        translated_label = LABEL_TRANSLATION.get(label_key, LABEL_TRANSLATION["Unknown"])
        print("\n" + "="*50)
        print(f" Изображение: {os.path.basename(image_path)}")
        print(f" Диагноз: {translated_label}")
        print(f" Уверенность: {result['score']*100:.2f}%")
        print("="*50)
        return translated_label, result['score']
    except Exception as e: # Обработка ошибок
        print(f"ВНИМАНИЕ!!!")
        print(f"Ошибка: {str(e)}")
        return None, 0.0

def evaluate_folder_images(folder_path, model_path): # Папка
    model = load_model(model_path)
    processor = ViTImageProcessor.from_pretrained(model_path)
    results = []
    for filename in os.listdir(folder_path): # Итерация по файлам
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(folder_path, filename)
            try:
                image = Image.open(image_path).convert("RGB")
                classifier = pipeline("image-classification",
                                    model=model,
                                    feature_extractor=processor)
                result = classifier(image)[0] # Сбор результатов в список
                label_key = result['label'].split('_')[-1]
                translated_label = LABEL_TRANSLATION.get(label_key, LABEL_TRANSLATION["Unknown"])
                print(f"\n Файл: {filename}")
                print(f" Результат: {translated_label}")
                print(f" Уверенность: {result['score']*100:.2f}%")
                print(f"\n")
                results.append((filename, translated_label, result['score']))
            except Exception as e:
                print(f"ВНИМАНИЕ!")
                print(f"Ошибка в файле {filename}: {str(e)}")
                results.append((filename, "Ошибка обработки", 0.0))
    return results

In [ ]:
evaluate_single_image("melanoma_attacks/pic/1.jpeg", "melanoma_attacks/checkpoint-12670")